<a href="https://colab.research.google.com/github/franjgs/Federated_Machine_Learning/blob/main/Federated_Skin_Lesion_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Introducción

## 1.1 Descripción del Proyecto

Este proyecto implementa un sistema de Aprendizaje Federado para la clasificación de lesiones cutáneas, centrándose en la privacidad de los datos del usuario. El objetivo principal es entrenar un modelo de clasificación de lesiones de piel utilizando datos de múltiples fuentes (dispositivos móviles), sin que los datos brutos de los pacientes salgan de los dispositivos.

### Arquitectura del Sistema

La arquitectura se compone de dos tipos de nodos principales:

*   **Nodos Edge (Aplicaciones Móviles)**: Representan las aplicaciones instaladas en los dispositivos de los usuarios. Cada aplicación es capaz de realizar inferencias locales sobre imágenes de lesiones cutáneas. En un flujo de Aprendizaje Federado, estas aplicaciones también generan "paquetes federados" (actualizaciones del modelo o características procesadas con privacidad diferencial) que se envían al servidor central. Los datos de las imágenes originales nunca abandonan el dispositivo del usuario.

*   **Servidor Central (Hospital)**: Actúa como el orquestador del sistema. Es responsable de entrenar un modelo global inicial utilizando un dataset centralizado (HAM10000 en este caso). Una vez entrenado, el servidor despliega este modelo a las aplicaciones móviles. Posteriormente, el servidor agrega los paquetes federados recibidos de los nodos Edge para actualizar iterativamente el modelo global, mejorando su rendimiento sin acceder directamente a los datos sensibles de los pacientes.

### Módulos Clave del Proyecto

El proyecto está organizado en varios módulos Python para facilitar la modularidad y la comprensión:

*   `src/utils/vision_logic.py`: Contiene las funciones esenciales para el procesamiento de imágenes dermatoscópicas. Incluye algoritmos para el preprocesamiento (equalización adaptativa del histograma, filtrado morfológico y de mediana), segmentación de la región de interés (usando Watershed y limpieza de bordes) y extracción de características (`ABCD` - Asimetría, Borde, Color, Diámetro) y texturales a partir de la lesión segmentada.

*   `src/hospital_server.py`: Implementa la lógica del servidor central. Esto incluye la construcción del dataset de entrenamiento a partir de las imágenes y metadatos, el entrenamiento del modelo de clasificación (SVM en este ejemplo) y su posterior despliegue (guardando el modelo y el escalador). También incluye una función `receive_federated_update` que simula la recepción y agregación de los paquetes federados de los nodos Edge.

*   `src/mobile_app.py`: Simula la funcionalidad de una aplicación móvil. Carga el modelo y el escalador desplegados por el servidor. Permite realizar inferencias sobre nuevas imágenes capturadas por el usuario. Además, implementa la funcionalidad `get_federated_packet` para crear un paquete de datos (características extraídas más un ruido diferencial para preservar la privacidad) y la etiqueta confirmada por el usuario (simulando la validación de un médico), que luego es enviado al servidor.

### Flujo de Aprendizaje Federado

El proceso general sigue estos pasos:

1.  **Entrenamiento Inicial en el Servidor**: El `HospitalServer` entrena un modelo inicial utilizando un dataset disponible localmente.
2.  **Despliegue del Modelo**: El modelo entrenado y su escalador se guardan (`global_model.pkl`, `global_scaler.pkl`) y se ponen a disposición de las aplicaciones móviles.
3.  **Inferencia Local en la App Móvil**: La `MobileApp` descarga el modelo, procesa una imagen del usuario, extrae características y realiza una predicción local.
4.  **Confirmación del Usuario y Generación de Paquete Federado**: Si el usuario (o un profesional) confirma la etiqueta de la lesión, la `MobileApp` genera un "paquete federado" que contiene las características extraídas (con ruido para privacidad diferencial) y la etiqueta confirmada.
5.  **Actualización Federada en el Servidor**: El `HospitalServer` recibe estos paquetes de múltiples aplicaciones y los utiliza para actualizar o mejorar su modelo global sin ver los datos brutos de cada usuario.

Este enfoque permite mejorar la precisión del modelo global a medida que más usuarios utilizan la aplicación y proporcionan retroalimentación, todo ello mientras se mantiene la privacidad de los datos individuales.

# 2. Configuración del Entorno

In [36]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [37]:
# Crear directorios
import os
# Definimos la ruta base en Google Drive para la estructura de módulos
BASE_COLAB_PATH = '/content/drive/MyDrive/Colab Notebooks/Federated_Learning'
os.makedirs(os.path.join(BASE_COLAB_PATH, 'src/utils'), exist_ok=True)
os.makedirs(os.path.join(BASE_COLAB_PATH, 'hospital_data'), exist_ok=True) # Directorio para almacenar el dataset HAM10000

# 3. Definición de Módulos

Esta sección contiene las definiciones de los módulos (`vision_logic.py`, `hospital_server.py`, `mobile_app.py`) que se utilizan en la arquitectura de aprendizaje federado. Se guardan en el sistema de archivos de Google Drive utilizando `%%writefile` para simular una estructura de proyecto modular.

## 3.1 Código para procesado de imagen (`src/utils/vision_logic.py`)
Este módulo contiene las funciones de procesamiento de imagen.

In [39]:
%%writefile "/content/drive/MyDrive/Colab Notebooks/Federated_Learning/src/utils/vision_logic.py"
import cv2
import numpy as np
from skimage import filters, morphology, segmentation, exposure, util, measure, color
from scipy import ndimage as ndi
from skimage.feature import graycomatrix, graycoprops

# Helper functions adapted from the notebook's global scope
def morph_closing_each(image, structuring_element):
    # Handle multi-channel images by applying operation to each channel
    if image.ndim == 3:
        closed_channels = [morphology.closing(image[:,:,c], structuring_element) for c in range(image.shape[2])]
        return np.stack(closed_channels, axis=-1)
    else:
        return morphology.closing(image, structuring_element)

def median_filter_each(image, footprint):
    # Handle multi-channel images by applying operation to each channel
    if image.ndim == 3:
        filtered_channels = [filters.median(image[:,:,c], footprint) for c in range(image.shape[2])]
        return np.stack(filtered_channels, axis=-1)
    else:
        return filters.median(image, footprint)

def preprocessing(image, structuring_element_val=5):
    # Ensure structuring_element is defined for this scope
    structuring_element = morphology.disk(structuring_element_val)

    equalized_adapthist = exposure.equalize_adapthist(image) # This can handle 3D images channel-wise
    img_morph_closing = morph_closing_each(equalized_adapthist, structuring_element)
    img_filtered = median_filter_each(img_morph_closing, structuring_element)
    return img_filtered

def images_segmentation(image):
    gray_img = color.rgb2gray(image)
    elevation_map = filters.sobel(gray_img)
    markers = np.zeros_like(gray_img, dtype=np.int32)
    threshold = filters.threshold_isodata(gray_img)
    markers[gray_img > threshold] = 1
    markers[gray_img < threshold] = 2
    segmented_img = segmentation.watershed(elevation_map, markers)
    segmented_img = ndi.binary_fill_holes(segmented_img - 1)
    segmented_img = morphology.remove_small_objects(segmented_img, min_size=800)
    img_border_cleared = segmentation.clear_border(segmented_img)
    labeled_img = morphology.label(img_border_cleared)
    props = measure.regionprops(labeled_img)
    num_labels = len(props)
    areas = [region.area for region in props]

    if num_labels > 0 and areas[np.argmax(areas)] >= 1200:
        target_label = props[np.argmax(areas)].label
    else:
        labeled_img = morphology.label(segmented_img)
        props = measure.regionprops(labeled_img)
        areas = [region.area for region in props]
        extents = [region.extent for region in props]
        region_max1 = np.argmax(areas)
        if len(props) > 1:
            areas_copy = areas.copy()
            areas_copy[region_max1] = 0
            region_max2 = np.argmax(areas_copy)
        else:
            region_max2 = -1
        if len(props) > 2:
            areas_copy[region_max2] = 0
            region_max3 = np.argmax(areas_copy)
        else:
            region_max3 = -1
        target_label = props[region_max1].label
        if extents[region_max1] > 0.50:
            target_label = props[region_max1].label
        elif region_max2 != -1 and extents[region_max2] > 0.50:
            target_label = props[region_max2].label
        elif region_max3 != -1 and extents[region_max3] > 0.50:
            target_label = props[region_max3].label
        else:
            target_label = props[region_max1].label

    for row, col in np.ndindex(labeled_img.shape):
        if labeled_img[row, col] != target_label:
            labeled_img[row, col] = 0
    for region in measure.regionprops(labeled_img):
        if region.label == target_label:
            return region
    return None

def extract_abcd_features_detailed(image, lesion_region):
    if lesion_region is None: # Handle cases where segmentation fails
        return np.zeros(11) # Return a vector of zeros or handle as error

    gray_img = color.rgb2gray(image)

    # 1] ASYMMETRY
    area_total = lesion_region.area
    img_mask = lesion_region.image

    horizontal_flip = np.fliplr(img_mask)
    diff_horizontal = img_mask * ~horizontal_flip

    vertical_flip = np.flipud(img_mask)
    diff_vertical = img_mask * ~vertical_flip

    diff_horizontal_area = np.count_nonzero(diff_horizontal)
    diff_vertical_area = np.count_nonzero(diff_vertical)
    asymm_idx = 0.5 * ((diff_horizontal_area / area_total) + (diff_vertical_area / area_total)) if area_total > 0 else 0
    ecc = lesion_region.eccentricity

    # 2] Border irregularity:
    compact_index = (lesion_region.perimeter ** 2) / (4 * np.pi * area_total) if area_total > 0 else 0

    # 3] Color variegation:
    # Ensure slicing doesn't go out of bounds or is empty
    # lesion_region.slice returns a tuple of slice objects, e.g., (slice(min_row, max_row, None), slice(min_col, max_col, None))
    min_row = lesion_region.slice[0].start
    max_row = lesion_region.slice[0].stop
    min_col = lesion_region.slice[1].start
    max_col = lesion_region.slice[1].stop

    if min_row >= max_row or min_col >= max_col:
        # Handle empty slice case, return default features
        return np.zeros(11)

    sliced = image[min_row:max_row, min_col:max_col]
    if sliced.shape[0] == 0 or sliced.shape[1] == 0:
        return np.zeros(11)

    lesion_r = sliced[:, :, 0]
    lesion_g = sliced[:, :, 1]
    lesion_b = sliced[:, :, 2]

    C_r = np.std(lesion_r) / np.max(lesion_r) if np.max(lesion_r) > 0 else 0
    C_g = np.std(lesion_g) / np.max(lesion_g) if np.max(lesion_g) > 0 else 0
    C_b = np.std(lesion_b) / np.max(lesion_b) if np.max(lesion_b) > 0 else 0

    # 4] Diameter:
    eq_diameter = lesion_region.equivalent_diameter

    # 5] Texture:
    glcm = graycomatrix(image=util.img_as_ubyte(gray_img), distances=[1],
                                angles=[0, np.pi/4, np.pi/2, np.pi * 3/2],
                                symmetric=True, normed=True)

    correlation = np.mean(graycoprops(glcm, prop='correlation'))
    homogeneity = np.mean(graycoprops(glcm, prop='homogeneity'))
    energy = np.mean(graycoprops(glcm, prop='energy'))
    contrast = np.mean(graycoprops(glcm, prop='contrast'))

    return np.array([asymm_idx, ecc, compact_index, C_r, C_g, C_b,
                    eq_diameter, correlation, homogeneity, energy, contrast])

def process_single_image_for_features_pipeline(image_path):
    try:
        img = cv2.imread(image_path)
        if img is None:
            print(f"Error: Could not load image from {image_path}")
            return None

        # Step 1: Preprocessing
        preprocessed_img = preprocessing(img)

        # Step 2: Segmentation
        segmented_region = images_segmentation(preprocessed_img)

        # Step 3: Feature Extraction
        if segmented_region:
            features = extract_abcd_features_detailed(preprocessed_img, segmented_region)
            return features
        else:
            print(f"Warning: No valid segmented region found for {image_path}")
            return np.zeros(11) # Return zero features if segmentation fails
    except Exception as e:
        print(f"Error processing image {image_path}: {e}")
        return np.zeros(11)


def get_skin_lesion_mask(image):
    # Original function, can keep for compatibility or simpler cases
    if len(image.shape) == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    else:
        gray = image
    val = filters.threshold_otsu(gray)
    mask = gray < val
    mask = morphology.closing(mask, morphology.disk(3))
    mask = ndi.binary_fill_holes(mask)
    return mask

def extract_abcd_features(image, mask):
    # Original function, can keep for compatibility or simpler cases
    area = np.sum(mask)
    perimetro = np.sum(segmentation.clear_border(mask))
    return [area, perimetro, np.mean(image[mask]), np.std(image[mask])]

Overwriting /content/drive/MyDrive/Colab Notebooks/Federated_Learning/src/utils/vision_logic.py


## 3.2 Módulo del Servidor del Hospital (`src/hospital_server.py`)

Este módulo implementa la lógica central del `HospitalServer`, que actúa como el orquestador del sistema de Aprendizaje Federado. Sus funciones principales incluyen:

*   **Inicialización (`__init__`)**: Carga los metadatos del dataset HAM10000 y, opcionalmente, las regiones segmentadas previamente procesadas de las imágenes de entrenamiento (provenientes de un archivo PKL). Esto evita la resegmentación costosa en cada ejecución. Inicializa un `StandardScaler` para la normalización de características y un modelo `SVC` (Support Vector Classifier) de scikit-learn con un kernel RBF y ajuste de pesos para clases desequilibradas.

*   **Construcción del Dataset (`build_dataset`)**: Procesa las imágenes del dataset HAM10000 utilizando las regiones segmentadas cargadas. Para cada imagen, realiza un preprocesamiento, extrae características detalladas ABCD (Asimetría, Borde, Color, Diámetro) y texturales utilizando las funciones definidas en `vision_logic.py`, y construye el conjunto de datos `X` (características) e `y` (etiquetas de diagnóstico) para el entrenamiento. Si las regiones segmentadas no están disponibles, el proceso no puede continuar.

*   **Entrenamiento y Despliegue (`train_and_deploy`)**: Si hay datos de entrenamiento disponibles, escala las características utilizando el `StandardScaler` y entrena el modelo `SVC`. Una vez entrenado, el modelo (`global_model.pkl`) y el escalador (`global_scaler.pkl`) se guardan en el sistema de archivos, simulando su despliegue a las aplicaciones móviles (`MobileApp`). Si no hay datos, se guardan archivos dummy para evitar errores, pero se notifica que no hubo entrenamiento real.

*   **Recepción de Actualizaciones Federadas (`receive_federated_update`)**: Esta función actúa como un *placeholder* crucial para la etapa de Aprendizaje Federado. Simula la recepción de

In [40]:
%%writefile "/content/drive/MyDrive/Colab Notebooks/Federated_Learning/src/hospital_server.py"
import pandas as pd
import joblib
from sklearn import svm
from sklearn.preprocessing import StandardScaler
import cv2
import numpy as np
import os
from src.utils.vision_logic import extract_abcd_features_detailed, preprocessing, images_segmentation, process_single_image_for_features_pipeline

class HospitalServer:
    def __init__(self, metadata_path, images_dir, segmented_regions_train_path=None):
        self.metadata = pd.read_csv(metadata_path)
        self.images_dir = images_dir
        self.scaler = StandardScaler()
        self.model = svm.SVC(kernel='rbf', probability=True, class_weight='balanced')
        self.segmented_regions_train = None
        if segmented_regions_train_path:
            print(f"Cargando regiones segmentadas de entrenamiento desde {segmented_regions_train_path}...")
            self.segmented_regions_train = joblib.load(segmented_regions_train_path)
            print(f"Cargadas {len(self.segmented_regions_train)} regiones segmentadas.")

    def build_dataset(self):
        """Procesa las imágenes de HAM10000 para crear el dataset de entrenamiento usando regiones segmentadas"""
        features_list = []
        labels = []

        if self.segmented_regions_train is None:
            print("Error: segmented_regions_train no ha sido cargado. No se puede construir el dataset.")
            return np.array([]), np.array([])

        print(f"Construyendo dataset desde {self.images_dir} con regiones segmentadas...")

        # Filtrar metadatos para incluir solo imágenes para las cuales se tienen regiones segmentadas
        metadata_for_training = self.metadata[self.metadata['image_id'].isin([img_name.replace('.jpg','') for img_name in self.segmented_regions_train.keys() if isinstance(img_name, str)])]
        print(f"Procesando {len(metadata_for_training)} imágenes para entrenamiento...")

        for idx, row in metadata_for_training.iterrows():
            image_id = row['image_id']
            img_name = f"{image_id}.jpg"
            img_path = os.path.join(self.images_dir, img_name)

            if img_name in self.segmented_regions_train:
                img = cv2.imread(img_path)
                if img is not None:
                    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # Convertir a RGB para funciones skimage
                    # Recuperar la región pre-segmentada del archivo .pkl cargado
                    lesion_region = self.segmented_regions_train[img_name]

                    # Realizar preprocesamiento y luego extraer características usando la región proporcionada
                    preprocessed_img = preprocessing(img_rgb) # El preprocesamiento sigue siendo necesario para una entrada consistente en la extracción de características
                    features = extract_abcd_features_detailed(preprocessed_img, lesion_region)

                    if features is not None and not np.array_equal(features, np.zeros(11)):
                        features_list.append(features)
                        labels.append(row['dx'])
                else:
                    print(f"Advertencia: No se pudo cargar la imagen {img_path}")
            else:
                print(f"Advertencia: Región segmentada para {img_name} no encontrada en el .pkl de entrenamiento.")

        print(f"Dataset construido con {len(features_list)} muestras.")
        return np.array(features_list), np.array(labels)

    def train_and_deploy(self, X, y):
        if len(X) == 0:
            print("No hay datos para entrenar el modelo.")
            # Se guardan archivos dummy para evitar errores posteriores, pero se indica que no hubo entrenamiento real
            joblib.dump(None, 'global_model.pkl')
            joblib.dump(None, 'global_scaler.pkl')
            print("Archivos de modelo y escalador dummy desplegados (sin entrenamiento real).")
            return

        X_scaled = self.scaler.fit_transform(X)
        self.model.fit(X_scaled, y)
        # Guardar archivos para la aplicación
        joblib.dump(self.model, 'global_model.pkl')
        joblib.dump(self.scaler, 'global_scaler.pkl')
        print("Modelo desplegado para Nodos Edge.")

    def receive_federated_update(self, federated_packets):
        """Placeholder para la función de actualización del modelo con paquetes federados"""
        print(f"HospitalServer recibió {len(federated_packets)} paquetes federados.")
        for packet in federated_packets:
            print(f"  - Características: {packet['features']}, Etiqueta confirmada: {packet['label']}")
        print("\nLa implementación de Federated Learning para actualizar el modelo se agrega aquí.")
        # En un sistema real, aquí se agregaría la lógica para promediar los pesos,
        # reentrenar parcialmente o almacenar los datos para un reentrenamiento periódico.

Overwriting /content/drive/MyDrive/Colab Notebooks/Federated_Learning/src/hospital_server.py


## 3.3 Módulo de la Aplicación Móvil (`src/mobile_app.py`)

Este módulo simula la funcionalidad de una aplicación móvil en el sistema de Aprendizaje Federado. Sus funciones principales incluyen:

*   **Inicialización (`__init__`)**: Carga el modelo de clasificación y el escalador de características (`global_model.pkl`, `global_scaler.pkl`) que han sido desplegados por el `HospitalServer`. Esto permite que la aplicación realice inferencias locales.

*   **Inferencia (`infer`)**: Procesa una imagen de lesión cutánea proporcionada por el usuario. Utiliza la pipeline de procesamiento de imágenes (`process_single_image_for_features_pipeline` de `vision_logic.py`) para extraer características. Si la extracción es exitosa, escala estas características y utiliza el modelo cargado para realizar una predicción local sobre la imagen.

*   **Generación de Paquete Federado (`get_federated_packet`)**: Después de una inferencia exitosa, esta función crea un "paquete federado" para enviar al servidor central. Este paquete contiene las características extraídas de la imagen (a las cuales se les añade un pequeño ruido aleatorio para preservar la privacidad diferencial de los datos del usuario) y la etiqueta de diagnóstico confirmada por el usuario (simulando la validación de un médico). Este paquete es esencial para la fase de actualización federada del modelo global.

In [41]:
%%writefile "/content/drive/MyDrive/Colab Notebooks/Federated_Learning/src/mobile_app.py"
import joblib
import cv2
import numpy as np
from src.utils.vision_logic import process_single_image_for_features_pipeline

class MobileApp:
    def __init__(self, model_path, scaler_path):
        self.model = joblib.load(model_path)
        self.scaler = joblib.load(scaler_path)
        self.last_vector = None

    def infer(self, image_path):
        if self.model is None or self.scaler is None:
            print("Error: Modelo o escalador no cargados. ¿Se entrenó el servidor correctamente?")
            return "Error"

        # Se utiliza la nueva pipeline de procesamiento de imágenes
        features = process_single_image_for_features_pipeline(image_path)

        if features is None or np.array_equal(features, np.zeros(len(features))): # Verificar si las características están vacías/cero
            print(f"Advertencia: No se pudieron extraer características válidas de {image_path}. No se realizará la inferencia.")
            return "No se pudo inferir (fallo en extracción de características)"

        self.last_vector = features # Se guardan las características extraídas

        # Predicción
        x_scaled = self.scaler.transform([self.last_vector])
        prediction = self.model.predict(x_scaled)[0]
        return prediction

    def get_federated_packet(self, confirmed_label):
        """Genera el paquete de datos para enviar al hospital (Federated Learning)"""
        if self.last_vector is None:
            print("Error: No se han extraído características todavía. Ejecutar infer() primero.")
            return None

        # Privacidad diferencial: se añade un pequeño ruido
        noise = np.random.normal(0, 0.01, len(self.last_vector))
        return {
            'features': self.last_vector + noise,
            'label': confirmed_label
        }

Overwriting /content/drive/MyDrive/Colab Notebooks/Federated_Learning/src/mobile_app.py


# 4. Ejecución del Flujo Completo (Main)
Aquí se conecta todo el sistema en Colab.

## 4.1. Preparación y Segmentación de Datos

Esta sección se encarga de la preparación de los datos necesarios para el entrenamiento del modelo y la simulación de la aplicación móvil. El proceso implica la carga de los metadatos del dataset HAM10000, la división en conjuntos de entrenamiento y prueba, y la segmentación de las lesiones en las imágenes correspondientes.

Es fundamental que el dataset HAM10000 (`HAM10000_metadata.csv` y las imágenes en el directorio `dataset`) haya sido descargado previamente y se encuentre en la ruta `/content/drive/MyDrive/ham10000/` en Google Drive. Las imágenes deben estar específicamente en `/content/drive/MyDrive/ham10000/dataset`.

Para optimizar el rendimiento y evitar la resegmentación de imágenes en cada ejecución, este paso comprueba si los archivos PKL con las regiones segmentadas de entrenamiento y prueba (en este caso, una muestra reducida para agilizar la demostración) ya existen. Si no existen, se procede a:

1.  Cargar el archivo de metadatos del HAM10000.
2.  Dividir una muestra del dataset (200 imágenes en este ejemplo para reproducibilidad y rapidez) en conjuntos de entrenamiento y prueba, asegurando una estratificación adecuada por tipo de lesión (`dx`).
3.  Iterar sobre las imágenes de cada conjunto, realizar un preprocesamiento y aplicar el algoritmo de segmentación (`images_segmentation`) para extraer la región de interés de la lesión.
4.  Guardar las regiones segmentadas resultantes en archivos PKL (`segmented_region_train_sample.pkl` y `segmented_region_test_sample.pkl`) en `/content/drive/MyDrive/ham10000/`.

**Nota:** Para la generación de los archivos PKL con *todas* las imágenes segmentadas del dataset HAM10000 (no solo la muestra utilizada aquí), se asume que el cuaderno `skin_lesions_classifier.ipynb` ha sido ejecutado previamente. Este proceso es clave para que las variables `segmented_train_regions` y `segmented_test_regions` estén disponibles y el `HospitalServer` pueda construir su dataset de entrenamiento de manera eficiente.


In [38]:
import pandas as pd
import joblib
import os
import cv2
from sklearn.model_selection import train_test_split
from src.utils.vision_logic import preprocessing, images_segmentation

# Rutas definidas previamente en el estado del cuaderno
METADATA_PATH = '/content/drive/MyDrive/ham10000/HAM10000_metadata.csv'
IMAGES_DIR = '/content/drive/MyDrive/ham10000/dataset'

# Rutas para los archivos PKL de la muestra (se guardarán en Google Drive)
SEGMENTED_REGIONS_TRAIN_SAMPLE_PATH = '/content/drive/MyDrive/ham10000/segmented_region_train_sample.pkl'
SEGMENTED_REGIONS_TEST_SAMPLE_PATH = '/content/drive/MyDrive/ham10000/segmented_region_test_sample.pkl'

# --- NUEVA COMPROBACIÓN: Si los archivos PKL ya existen, saltar la generación ---
if os.path.exists(SEGMENTED_REGIONS_TRAIN_SAMPLE_PATH) and os.path.exists(SEGMENTED_REGIONS_TEST_SAMPLE_PATH):
    print("--- Archivos PKL de regiones segmentadas (muestra) ya existen. Saltando la generación. ---")
    # Cargar para que las variables segmented_train_regions y segmented_test_regions estén disponibles para futuras referencias
    segmented_train_regions = joblib.load(SEGMENTED_REGIONS_TRAIN_SAMPLE_PATH)
    segmented_test_regions = joblib.load(SEGMENTED_REGIONS_TEST_SAMPLE_PATH)
    print(f"Cargadas {len(segmented_train_regions)} regiones segmentadas de entrenamiento desde {SEGMENTED_REGIONS_TRAIN_SAMPLE_PATH}")
    print(f"Cargadas {len(segmented_test_regions)} regiones segmentadas de prueba desde {SEGMENTED_REGIONS_TEST_SAMPLE_PATH}")
else:
    print("--- GENERANDO ARCHIVOS PKL DE REGIONES SEGMENTADAS (MUESTRA) ---")

    # Cargar metadatos
    metadata = pd.read_csv(METADATA_PATH)

    # Dividir metadatos en entrenamiento y prueba
    # Se utiliza un pequeño subconjunto y un estado aleatorio fijo para reproducibilidad y una demostración más rápida.
    # La configuración del tamaño de la muestra y de prueba puede ajustarse para un entrenamiento real.
    if len(metadata) > 200: # Si el dataset es grande, se toma una muestra menor para velocidad de demostración
        sample_metadata = metadata.sample(n=200, random_state=42).reset_index(drop=True)
    else:
        sample_metadata = metadata

    train_df, test_df = train_test_split(sample_metadata, test_size=0.2, random_state=42, stratify=sample_metadata['dx'])

    # Procesar imágenes de entrenamiento
    segmented_train_regions = {}
    print(f"Procesando {len(train_df)} imágenes de entrenamiento para segmentación...")
    for idx, row in train_df.iterrows():
        image_id = row['image_id']
        img_name = f"{image_id}.jpg"
        img_path = os.path.join(IMAGES_DIR, img_name)

        if os.path.exists(img_path):
            img = cv2.imread(img_path)
            if img is not None:
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                preprocessed_img = preprocessing(img_rgb)
                segmented_region = images_segmentation(preprocessed_img)
                if segmented_region:
                    segmented_train_regions[img_name] = segmented_region
            else:
                print(f"Advertencia: No se pudo cargar la imagen {img_path}")
        else:
            print(f"Advertencia: Archivo no encontrado para la imagen {img_path}")

    # Procesar imágenes de prueba
    segmented_test_regions = {}
    print(f"Procesando {len(test_df)} imágenes de prueba para segmentación...")
    for idx, row in test_df.iterrows():
        image_id = row['image_id']
        img_name = f"{image_id}.jpg"
        img_path = os.path.join(IMAGES_DIR, img_name)

        if os.path.exists(img_path):
            img = cv2.imread(img_path)
            if img is not None:
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                preprocessed_img = preprocessing(img_rgb)
                segmented_region = images_segmentation(preprocessed_img)
                if segmented_region:
                    segmented_test_regions[img_name] = segmented_region
            else:
                print(f"Advertencia: No se pudo cargar la imagen {img_path}")
        else:
            print(f"Advertencia: Archivo no encontrado para la imagen {img_path}")

    # Guardar regiones segmentadas en archivos .pkl de muestra
    joblib.dump(segmented_train_regions, SEGMENTED_REGIONS_TRAIN_SAMPLE_PATH)
    joblib.dump(segmented_test_regions, SEGMENTED_REGIONS_TEST_SAMPLE_PATH)

    print(f"Guardadas {len(segmented_train_regions)} regiones segmentadas de entrenamiento en {SEGMENTED_REGIONS_TRAIN_SAMPLE_PATH}")
    print(f"Guardadas {len(segmented_test_regions)} regiones segmentadas de prueba en {SEGMENTED_REGIONS_TEST_SAMPLE_PATH}")
    print("--- ARCHIVOS PKL DE REGIONES SEGMENTADAS (MUESTRA) GENERADOS ---")

--- Archivos PKL de regiones segmentadas (muestra) ya existen. Saltando la generación. ---
Cargadas 160 regiones segmentadas de entrenamiento desde /content/drive/MyDrive/ham10000/segmented_region_train_sample.pkl
Cargadas 40 regiones segmentadas de prueba desde /content/drive/MyDrive/ham10000/segmented_region_test_sample.pkl


## 4.2. Preparar una imagen de prueba
Para que la simulación de la aplicación móvil sea ejecutable sin necesidad de subir una foto, se creará una imagen de prueba genérica.

In [42]:
import cv2
import numpy as np

# Se crea una imagen de prueba simple
dummy_image_path = 'dummy_skin_lesion.jpg'
dummy_image = np.zeros((100, 100, 3), dtype=np.uint8)
# Se dibuja un círculo para simular una lesión
cv2.circle(dummy_image, (50, 50), 20, (0, 0, 255), -1) # Círculo rojo
cv2.imwrite(dummy_image_path, dummy_image)

print(f"Imagen de prueba creada en: {dummy_image_path}")


Imagen de prueba creada en: dummy_skin_lesion.jpg


##4.3 Ejecución del Flujo Completo (Main) del Sistema de Aprendizaje Federado

Este script integra y ejecuta el flujo completo del sistema de Aprendizaje Federado para la clasificación de lesiones cutáneas, tal como se define en los módulos `hospital_server.py` y `mobile_app.py`, y utiliza las utilidades de `vision_logic.py`.

## 1. Configuración del Entorno y Carga de Módulos

-   **Importaciones**: Se importan las librerías necesarias como `numpy`, `joblib`, `os`, `importlib` y `sys`.
-   **Ruta Base**: Se añade `BASE_COLAB_PATH` (`/content/drive/MyDrive/Colab Notebooks/Federated_Learning`) al `sys.path` para que Python pueda localizar los módulos personalizados (`src/hospital_server.py`, `src/mobile_app.py`, `src/utils/vision_logic.py`).
-   **Recarga de Módulos**: Se utilizan `importlib.reload()` para forzar la recarga de los módulos. Esto es crucial en entornos interactivos como Colab para asegurar que cualquier cambio en los archivos de los módulos (`%%writefile`) se aplique antes de su uso.
-   **Instanciación de Clases**: Se importan `HospitalServer` y `MobileApp` de sus respectivos archivos.

## 2. Configuración de Rutas de Datos

Se definen las rutas esenciales para el funcionamiento del sistema:

-   `METADATA_PATH`: Ruta al archivo CSV con los metadatos del dataset HAM10000.
-   `IMAGES_DIR`: Directorio que contiene las imágenes del dataset HAM10000.
-   `SEGMENTED_REGIONS_TRAIN_PATH` y `SEGMENTED_REGIONS_TEST_PATH`: Rutas a los archivos `.pkl` que contienen las regiones segmentadas de las imágenes de entrenamiento y prueba. Estos archivos son generados previamente para optimizar el rendimiento y evitar la resegmentación en cada ejecución.

## 3. Entrenamiento y Despliegue del Hospital

Esta sección simula el rol del servidor central (hospital) en el flujo de Aprendizaje Federado.

-   **Instanciación del Servidor**: Se crea una instancia de `HospitalServer`, pasándole las rutas a los metadatos, el directorio de imágenes y las regiones segmentadas de entrenamiento.
-   **Construcción del Dataset**: Se llama al método `build_dataset()` del servidor para procesar las imágenes de entrenamiento (usando las regiones segmentadas precalculadas) y extraer las características (`X_train`) y las etiquetas (`y_train`).
-   **Entrenamiento y Despliegue**: El servidor entrena su modelo de clasificación (un SVM en este caso) utilizando `X_train` y `y_train`. Una vez entrenado, el modelo y el escalador de características se guardan en archivos (`global_model.pkl`, `global_scaler.pkl`), simulando su despliegue a las aplicaciones móviles.

## 4. Simulación de la Aplicación Móvil (Inferencia Local)

Esta parte del script emula el comportamiento de una aplicación móvil en un dispositivo de usuario (nodo Edge).

-   **Instanciación de la App Móvil**: Se crea una instancia de `MobileApp`, cargando el modelo y el escalador que fueron desplegados por el `HospitalServer`.
-   **Imagen de Prueba**: Se utiliza `dummy_image_path` (una imagen simple generada previamente) para simular una foto tomada por el usuario.
-   **Inferencia Local**: La aplicación llama a su método `infer()` para procesar la imagen de prueba, extraer características y realizar una predicción local usando el modelo descargado. El resultado de la predicción se imprime.

## 5. Simulación del Aprendizaje Federado

Finalmente, se simula el paso donde la aplicación móvil contribuye a la actualización del modelo global del servidor.

-   **Generación del Paquete Federado**: Si la inferencia local fue exitosa, se simula que un médico (o usuario) confirma la etiqueta de la lesión (ej. 'mel'). La `MobileApp` genera un `paquete federado`, que incluye las características extraídas (con ruido diferencial para privacidad) y la etiqueta confirmada.
-   **Recepción y Agregación en el Servidor**: El `HospitalServer` recibe este paquete (y potencialmente muchos otros de diferentes usuarios) a través de `receive_federated_update()`. Actualmente, esta función es un *placeholder* y en una implementación completa contendría la lógica para agregar los paquetes y actualizar el modelo global del servidor de manera federada.

Este flujo demuestra cómo un modelo se entrena centralmente, se despliega a clientes, los clientes realizan inferencias locales y, posteriormente, contribuyen con datos de características (con privacidad diferencial) y etiquetas confirmadas para futuras actualizaciones federadas del modelo central.

In [43]:
import numpy as np
import joblib # Importar joblib para cargar pkls
import os
import importlib
import sys

# Añadir la ruta base de los módulos a sys.path para que Python pueda encontrarlos
BASE_COLAB_PATH = '/content/drive/MyDrive/Colab Notebooks/Federated_Learning'
if BASE_COLAB_PATH not in sys.path:
    sys.path.insert(0, BASE_COLAB_PATH)

# Forzar la recarga de módulos para asegurar que se apliquen los últimos cambios
if 'src.utils.vision_logic' in sys.modules:
    importlib.reload(sys.modules['src.utils.vision_logic'])
if 'src.hospital_server' in sys.modules:
    importlib.reload(sys.modules['src.hospital_server'])
if 'src.mobile_app' in sys.modules:
    importlib.reload(sys.modules['src.mobile_app'])

from src.hospital_server import HospitalServer
from src.mobile_app import MobileApp


# 1. CONFIGURACIÓN DE RUTAS (Ajustar según la configuración de Drive)
METADATA_PATH = '/content/drive/MyDrive/ham10000/HAM10000_metadata.csv'
IMAGES_DIR = '/content/drive/MyDrive/ham10000/dataset' # Directorio de imágenes original

# Rutas a los archivos PKL con regiones segmentadas (se asume que ya existen)
# Ahora se usan los archivos de muestra generados por new_cell_pkl_gen
SEGMENTED_REGIONS_TRAIN_PATH = '/content/drive/MyDrive/ham10000/segmented_region_train_sample.pkl'
SEGMENTED_REGIONS_TEST_PATH = '/content/drive/MyDrive/ham10000/segmented_region_test_sample.pkl'

# Se asume que los archivos PKL de muestra se generaron previamente
# (por la celda 'e83b8a36') y por lo tanto existen.

# 2. ENTRENAMIENTO DEL HOSPITAL (Se realiza una única vez con datos reales)
print("\n--- INICIANDO ENTRENAMIENTO DEL HOSPITAL ---")
server = HospitalServer(METADATA_PATH, IMAGES_DIR, segmented_regions_train_path=SEGMENTED_REGIONS_TRAIN_PATH)

# Construir el dataset real
X_train, y_train = server.build_dataset()

# Entrenar y desplegar el modelo
server.train_and_deploy(X_train, y_train)
print("--- ENTRENAMIENTO DEL HOSPITAL FINALIZADO ---")


# 3. USO DE LA APLICACIÓN MÓVIL (Edge Computing)
print("\n--- SIMULANDO APP MÓVIL ---")
# La aplicación carga los archivos .pkl generados por el servidor
# Los archivos del modelo y scaler también se guardarán en la nueva ruta si se especifican ahí.
# Para simplificar, se asume que 'global_model.pkl' y 'global_scaler.pkl' se guardan en el directorio raíz
# donde se ejecuta el notebook, pero se podrían mover al BASE_COLAB_PATH si fuera necesario.
app = MobileApp('global_model.pkl', 'global_scaler.pkl')

# Se simula que el usuario realiza una foto con la imagen de prueba
# dummy_image_path se definió en una celda anterior, se debe asegurar que exista y sea accesible.
print(f"Ruta de la imagen de prueba para inferencia: {dummy_image_path}")

# Se puede usar una imagen real del dataset HAM10000 si está configurado
# Por ejemplo:
# real_image_for_inference = os.path.join(IMAGES_DIR, 'ISIC_0024306.jpg') # Reemplazar con un ID de imagen real
# print(f"Resultado predicción móvil con imagen real: {app.infer(real_image_for_inference)}")

# Para la demostración, se utilizará la imagen dummy
prediction_result = app.infer(dummy_image_path)
print(f"Resultado predicción móvil: {prediction_result}")
print("--- SIMULACIÓN APP MÓVIL FINALIZADA ---")


# 4. APRENDIZAJE FEDERADO
print("\n--- SIMULANDO APRENDIZAJE FEDERADO ---")
if prediction_result != "No se pudo inferir (fallo en extracción de características)" and app.last_vector is not None:
    # El médico confirma la etiqueta como 'mel' (ejemplo)
    confirmed_label = 'mel'
    print(f"El médico confirma la etiqueta como: {confirmed_label}")
    paquete = app.get_federated_packet(confirmed_label)

    if paquete is not None:
        # El hospital recibe la actualización de este usuario (y muchos otros)
        server.receive_federated_update([paquete])
    else:
        print("No se pudo generar el paquete federado.")
else:
    print("No se puede generar paquete federado debido a un fallo previo en la inferencia.")
print("--- SIMULACIÓN APRENDIZAJE FEDERADO FINALIZADA ---")


--- INICIANDO ENTRENAMIENTO DEL HOSPITAL ---
Cargando regiones segmentadas de entrenamiento desde /content/drive/MyDrive/ham10000/segmented_region_train_sample.pkl...
Cargadas 160 regiones segmentadas.
Construyendo dataset desde /content/drive/MyDrive/ham10000/dataset con regiones segmentadas...
Procesando 160 imágenes para entrenamiento...
Dataset construido con 160 muestras.
Modelo desplegado para Nodos Edge.
--- ENTRENAMIENTO DEL HOSPITAL FINALIZADO ---

--- SIMULANDO APP MÓVIL ---
Ruta de la imagen de prueba para inferencia: dummy_skin_lesion.jpg
Resultado predicción móvil: bkl
--- SIMULACIÓN APP MÓVIL FINALIZADA ---

--- SIMULANDO APRENDIZAJE FEDERADO ---
El médico confirma la etiqueta como: mel
HospitalServer recibió 1 paquetes federados.
  - Características: [ 1.90840940e-02 -8.41856831e-04  1.24724891e+00  2.17208673e-01
  2.25964937e-01  3.21484893e-01  1.12838949e+02  9.88805905e-01
  9.25921311e-01  7.25383786e-01  1.16717717e+00], Etiqueta confirmada: mel

La implementació